# BirdCLEF 2026 — EfficientNet-B2 + MixUp + SpecAugment + Focal Loss

**Key differences from teammates:**
- **EfficientNet-B2** backbone (stronger than B0) — loaded offline from Kaggle weights dataset
- **MixUp augmentation** on mel spectrograms
- **SpecAugment** (time + frequency masking)
- **Focal Loss** (handles class imbalance, not BCE)
- **Median-based robust normalization** (different from min-max and dynamic range approaches)
- **Soundscape data** with proper 5-sec chunk extraction
- **Label smoothing** to avoid overconfident predictions

**Required Kaggle datasets to add (Internet OFF):**
- `timm/efficientnet` → adds EfficientNet-B2 weights at `/kaggle/input/efficientnet/pytorch/b2/1/`
  - Search 'efficientnet b2 timm' on Kaggle Datasets and add the one with .pth weights
  - OR add dataset `thedrcat/timm-efficientnet-b2` which is widely used in BirdCLEF

Expected runtime: ~4–5h on T4 GPU

In [ ]:
!ls ../input/competitions/birdclef-2026

In [ ]:
import os
import ast
import time
import glob
import random
import torch
import torchaudio
import torchaudio.transforms as T
import pandas as pd
import numpy as np
import torch.nn as nn
import timm
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm import tqdm

BASE = "../input/competitions/birdclef-2026/"
TRAIN_AUDIO_DIR = os.path.join(BASE, "train_audio")
SOUNDSCAPE_DIR  = os.path.join(BASE, "train_soundscapes")
TRAIN_CSV       = os.path.join(BASE, "train.csv")
TAXONOMY_CSV    = os.path.join(BASE, "taxonomy.csv")
SOUNDSCAPE_CSV  = os.path.join(BASE, "train_soundscapes_labels.csv")
WORK_DIR        = "/kaggle/working/"

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

seed_everything(42)

class Config:
    SR          = 32000
    DURATION    = 5
    MAX_LENGTH  = SR * DURATION   # 160 000 samples
    N_MELS      = 128
    N_FFT       = 1024
    HOP_LENGTH  = 512
    BATCH_SIZE  = 32
    NUM_WORKERS = 4
    EPOCHS      = 12
    LR          = 8e-4
    WEIGHT_DECAY= 1e-4
    # MixUp
    MIXUP_ALPHA = 0.4
    MIXUP_PROB  = 0.5
    # SpecAugment
    FREQ_MASK_PARAM = 20   # max freq bins masked
    TIME_MASK_PARAM = 40   # max time frames masked
    N_FREQ_MASKS    = 2
    N_TIME_MASKS    = 2
    # Focal loss
    FOCAL_GAMMA  = 2.0
    FOCAL_ALPHA  = 0.25
    LABEL_SMOOTH = 0.05
    # Model
    MODEL_NAME   = 'efficientnet_b2'

taxonomy_df  = pd.read_csv(TAXONOMY_CSV)
CLASSES      = taxonomy_df['primary_label'].unique().tolist()
NUM_CLASSES  = len(CLASSES)
class_to_idx = {c: i for i, c in enumerate(CLASSES)}

print(f"Number of classes: {NUM_CLASSES}")

## Data manifest — train_audio + soundscapes

In [ ]:
def to_seconds(ts):
    if isinstance(ts, (int, float)):
        return float(ts)
    parts = str(ts).split(":")
    if len(parts) == 3:
        return int(parts[0]) * 3600 + int(parts[1]) * 60 + float(parts[2])
    if len(parts) == 2:
        return int(parts[0]) * 60 + float(parts[1])
    return float(ts)

train_df      = pd.read_csv(TRAIN_CSV)
soundscape_df = pd.read_csv(SOUNDSCAPE_CSV)

# --- train_audio rows ---
train_rows = []
for _, row in train_df.iterrows():
    labels = [row['primary_label']]
    sec_labels = ast.literal_eval(row.get('secondary_labels', "[]"))
    labels.extend(sec_labels)
    train_rows.append({
        'filename'    : row['filename'],
        'audio_path'  : os.path.join(TRAIN_AUDIO_DIR, row['filename']),
        'all_labels'  : list(set(labels)),
        'is_soundscape': 0,
        'start_sec'   : -1.0   # -1 means random crop
    })

# --- soundscape rows ---
soundscape_rows = []
for _, row in soundscape_df.iterrows():
    labels = [lab for lab in str(row['primary_label']).split(';') if lab in class_to_idx]
    if not labels:
        continue
    soundscape_rows.append({
        'filename'    : row['filename'],
        'audio_path'  : os.path.join(SOUNDSCAPE_DIR, row['filename']),
        'all_labels'  : labels,
        'is_soundscape': 1,
        'start_sec'   : to_seconds(row['start'])
    })

clean_manifest     = pd.DataFrame(train_rows)
soundscape_manifest = pd.DataFrame(soundscape_rows)

# Oversample soundscapes 2x (they are high-quality in-domain labels)
full_manifest = pd.concat(
    [clean_manifest, soundscape_manifest, soundscape_manifest],
    ignore_index=True
)

# Group split by filename to avoid leakage
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(gss.split(full_manifest, groups=full_manifest['filename']))

train_split = full_manifest.iloc[train_idx].reset_index(drop=True)
val_split   = full_manifest.iloc[val_idx].reset_index(drop=True)

print(f"Train size: {len(train_split)} | Val size: {len(val_split)}")

## Dataset with robust normalization + SpecAugment

In [ ]:
class BirdCLEFDataset(Dataset):
    """
    Key design choices vs teammates:
    - Robust median/IQR normalisation instead of min-max
    - SpecAugment (freq + time masking) applied on-the-fly
    - Exact start_sec crop for soundscapes, random crop for train_audio
    """

    def __init__(self, df, config, is_train=True):
        self.df       = df
        self.config   = config
        self.is_train = is_train

        self.mel_transform = T.MelSpectrogram(
            sample_rate = config.SR,
            n_fft       = config.N_FFT,
            hop_length  = config.HOP_LENGTH,
            n_mels      = config.N_MELS,
            f_min       = 50,
            f_max       = 14000,
            power       = 2.0
        )
        self.amp_to_db = T.AmplitudeToDB(stype='power', top_db=80)

        # SpecAugment transforms
        self.freq_masking = T.FrequencyMasking(freq_mask_param=config.FREQ_MASK_PARAM)
        self.time_masking = T.TimeMasking(time_mask_param=config.TIME_MASK_PARAM)

    def __len__(self):
        return len(self.df)

    def _load_and_crop(self, row):
        waveform, sr = torchaudio.load(row['audio_path'])

        # Mix to mono
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)

        # Resample if needed
        if sr != self.config.SR:
            waveform = torchaudio.functional.resample(waveform, sr, self.config.SR)

        audio_len = waveform.shape[1]
        target_len = self.config.MAX_LENGTH

        if audio_len >= target_len:
            if row['start_sec'] >= 0:  # soundscape: fixed crop
                start = int(row['start_sec'] * self.config.SR)
                start = min(start, audio_len - target_len)
            elif self.is_train:        # train_audio: random crop
                start = np.random.randint(0, audio_len - target_len + 1)
            else:                      # val: center crop
                start = (audio_len - target_len) // 2
            waveform = waveform[:, start:start + target_len]
        else:
            # Repeat-pad so the full clip fills the window
            n_repeat = (target_len // audio_len) + 1
            waveform  = waveform.repeat(1, n_repeat)[:, :target_len]

        return waveform

    def _to_mel(self, waveform):
        mel = self.mel_transform(waveform)          # (1, n_mels, T)
        mel = self.amp_to_db(mel)                   # log scale
        return mel

    def _robust_normalize(self, mel):
        """
        Subtract median, divide by IQR.
        More robust than min-max which is sensitive to a single loud spike.
        """
        flat = mel.flatten()
        median = flat.median()
        q75 = torch.quantile(flat, 0.75)
        q25 = torch.quantile(flat, 0.25)
        iqr  = q75 - q25 + 1e-6
        mel  = (mel - median) / iqr
        # Clip to keep range sane for the network
        mel  = mel.clamp(-4, 4)
        return mel

    def _specaugment(self, mel):
        for _ in range(self.config.N_FREQ_MASKS):
            mel = self.freq_masking(mel)
        for _ in range(self.config.N_TIME_MASKS):
            mel = self.time_masking(mel)
        return mel

    def _make_target(self, row):
        target = torch.zeros(NUM_CLASSES, dtype=torch.float32)
        for label in row['all_labels']:
            idx = class_to_idx.get(label)
            if idx is not None:
                target[idx] = 1.0
        return target

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        waveform = self._load_and_crop(row)
        mel      = self._to_mel(waveform)           # (1, n_mels, T)
        mel      = self._robust_normalize(mel)

        if self.is_train:
            mel  = self._specaugment(mel)

        # Repeat single channel → 3-channel for ImageNet pretrained backbone
        mel = mel.expand(3, -1, -1)                 # (3, n_mels, T)

        target = self._make_target(row)
        return mel, target

In [ ]:
train_dataset = BirdCLEFDataset(train_split, Config, is_train=True)
val_dataset   = BirdCLEFDataset(val_split,   Config, is_train=False)

train_loader = DataLoader(
    train_dataset,
    batch_size  = Config.BATCH_SIZE,
    shuffle     = True,
    num_workers = Config.NUM_WORKERS,
    pin_memory  = True,
    drop_last   = True
)

val_loader = DataLoader(
    val_dataset,
    batch_size  = Config.BATCH_SIZE,
    shuffle     = False,
    num_workers = Config.NUM_WORKERS,
    pin_memory  = True
)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

## Model — EfficientNet-B2

In [ ]:
# ── Offline weight loading for Kaggle (no internet) ──────────────────────────
# Add one of these public datasets to your notebook before running:
#   Option A (recommended): Kaggle Models → timm/efficientnet/pytorch/b2/1
#     go to kaggle.com/models/timm/efficientnet, select pytorch/b2/1, add to notebook
#   Option B: search 'efficientnet b2 timm' on kaggle.com/datasets
#
# The cell below auto-detects the weights path and loads offline.
# Falls back to efficientnet_b0 (already cached in docker) if not found.

import glob as _glob

def find_b2_weights():
    patterns = [
        # Kaggle Models: timm/efficientnet/pytorch/b2/1
        "/kaggle/input/efficientnet/pytorch/b2/1/*.pth",
        "/kaggle/input/efficientnet/pytorch/b2/1/*.bin",
        "/kaggle/input/efficientnet/pytorch/b2/1/*.safetensors",
        # Common dataset slugs used in BirdCLEF
        "/kaggle/input/timm-efficientnet-b2*/**/*.pth",
        "/kaggle/input/timm-efficientnet-b2*/**/*.bin",
        "/kaggle/input/pytorch-efficientnet-b2*/**/*.pth",
        "/kaggle/input/efficientnetb2*/**/*.pth",
        # HF cache (if weights were downloaded when internet was on)
        os.path.expanduser("~/.cache/huggingface/hub/*efficientnet_b2*/**/model.safetensors"),
    ]
    for pat in patterns:
        hits = _glob.glob(pat, recursive=True)
        if hits:
            return hits[0]
    return None


class BirdCLEFModel(nn.Module):
    def __init__(self, model_name, num_classes=NUM_CLASSES, pretrained_path=None):
        super().__init__()
        self.backbone = timm.create_model(
            model_name,
            pretrained  = False,
            num_classes = 0,
            drop_rate   = 0.3,
        )
        if pretrained_path:
            print(f"Loading backbone weights from: {pretrained_path}")
            state = torch.load(pretrained_path, map_location='cpu', weights_only=True)
            if isinstance(state, dict) and 'model' in state:
                state = state['model']
            missing, unexpected = self.backbone.load_state_dict(state, strict=False)
            print(f"  Missing keys: {len(missing)} | Unexpected keys: {len(unexpected)}")
        else:
            print("WARNING: No pretrained weights loaded.")

        in_features = self.backbone.num_features
        self.head = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(in_features, num_classes)
        )

    def forward(self, x):
        return self.head(self.backbone(x))


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

weights_path = find_b2_weights()
if weights_path:
    print(f"Found EfficientNet-B2 weights: {weights_path}")
    model = BirdCLEFModel('efficientnet_b2', pretrained_path=weights_path).to(device)
    backbone_name = 'efficientnet_b2'
else:
    print("B2 weights not found — using efficientnet_b0 (cached in Kaggle docker).")
    print("TIP: Add Kaggle Model 'timm/efficientnet/pytorch/b2/1' to use B2.")
    # B0 with pretrained=True works because it's cached in the docker image
    b0_backbone = timm.create_model('efficientnet_b0', pretrained=True, num_classes=0, drop_rate=0.3)
    model = BirdCLEFModel('efficientnet_b0')
    # Replace backbone with pretrained one
    model.backbone = b0_backbone
    model = model.to(device)
    backbone_name = 'efficientnet_b0'

print(f"Backbone: {backbone_name} | Device: {device}")
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {total_params:,}")


## Focal Loss + MixUp helpers

In [ ]:
class FocalLoss(nn.Module):
    """
    Binary Focal Loss for multi-label classification.
    Focuses training on hard, misclassified examples.
    gamma=2.0 is standard; alpha handles class imbalance.
    """
    def __init__(self, gamma=Config.FOCAL_GAMMA, alpha=Config.FOCAL_ALPHA,
                 label_smooth=Config.LABEL_SMOOTH, reduction='mean'):
        super().__init__()
        self.gamma        = gamma
        self.alpha        = alpha
        self.label_smooth = label_smooth
        self.reduction    = reduction

    def forward(self, logits, targets):
        # Label smoothing
        targets = targets * (1 - self.label_smooth) + self.label_smooth / 2

        bce   = nn.functional.binary_cross_entropy_with_logits(
            logits, targets, reduction='none'
        )
        probs = torch.sigmoid(logits)
        pt    = torch.where(targets > 0.5, probs, 1 - probs)
        alpha = torch.where(targets > 0.5,
                            torch.tensor(self.alpha, device=logits.device),
                            torch.tensor(1 - self.alpha, device=logits.device))
        focal = alpha * ((1 - pt) ** self.gamma) * bce

        if self.reduction == 'mean':
            return focal.mean()
        return focal.sum()


def mixup_data(x, y, alpha=Config.MIXUP_ALPHA):
    """Returns mixed inputs, pairs of targets, and lambda."""
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1.0
    batch_size = x.size(0)
    index      = torch.randperm(batch_size, device=x.device)
    mixed_x    = lam * x + (1 - lam) * x[index]
    y_a, y_b   = y, y[index]
    return mixed_x, y_a, y_b, lam


def mixup_criterion(criterion, logits, y_a, y_b, lam):
    return lam * criterion(logits, y_a) + (1 - lam) * criterion(logits, y_b)


def calculate_competition_roc_auc(y_true, y_pred):
    aucs = []
    for i in range(y_true.shape[1]):
        if len(np.unique(y_true[:, i])) == 2:
            aucs.append(roc_auc_score(y_true[:, i], y_pred[:, i]))
    return float(np.mean(aucs)) if aucs else 0.5

## Training

In [ ]:
criterion = FocalLoss()
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr           = Config.LR,
    weight_decay = Config.WEIGHT_DECAY
)
scheduler = CosineAnnealingLR(optimizer, T_max=Config.EPOCHS, eta_min=1e-6)

best_val_auc = 0.0
MODEL_SAVE_PATH = os.path.join(WORK_DIR, f'best_birdclef_{backbone_name}.pth')

for epoch in range(Config.EPOCHS):
    t0 = time.time()

    # ---- TRAIN ----
    model.train()
    train_loss = 0.0
    train_pbar = tqdm(train_loader, desc=f"Ep {epoch+1}/{Config.EPOCHS} [Train]", leave=False)

    for images, targets in train_pbar:
        images  = images.to(device)
        targets = targets.to(device)

        # Apply MixUp with probability MIXUP_PROB
        if np.random.rand() < Config.MIXUP_PROB:
            images, targets_a, targets_b, lam = mixup_data(images, targets)
            optimizer.zero_grad()
            logits = model(images)
            loss   = mixup_criterion(criterion, logits, targets_a, targets_b, lam)
        else:
            optimizer.zero_grad()
            logits = model(images)
            loss   = criterion(logits, targets)

        loss.backward()
        # Gradient clipping for stability
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        train_loss += loss.item() * images.size(0)
        train_pbar.set_postfix(loss=f"{loss.item():.4f}")

    train_loss /= len(train_loader.dataset)

    # ---- VALIDATE ----
    model.eval()
    val_loss       = 0.0
    all_val_targets = []
    all_val_preds   = []
    val_pbar = tqdm(val_loader, desc=f"Ep {epoch+1}/{Config.EPOCHS} [Val]", leave=False)

    with torch.no_grad():
        for images, targets in val_pbar:
            images  = images.to(device)
            targets = targets.to(device)
            logits  = model(images)
            loss    = criterion(logits, targets)
            val_loss += loss.item() * images.size(0)
            probs = torch.sigmoid(logits)
            all_val_targets.append(targets.cpu().numpy())
            all_val_preds.append(probs.cpu().numpy())
            val_pbar.set_postfix(loss=f"{loss.item():.4f}")

    val_loss /= len(val_loader.dataset)
    all_val_targets = np.vstack(all_val_targets)
    all_val_preds   = np.vstack(all_val_preds)
    val_auc = calculate_competition_roc_auc(all_val_targets, all_val_preds)

    current_lr = optimizer.param_groups[0]['lr']
    scheduler.step()

    elapsed = (time.time() - t0) / 60
    print(f"Epoch {epoch+1}/{Config.EPOCHS} | LR: {current_lr:.2e} | Time: {elapsed:.1f}min")
    print(f"  -> Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val ROC-AUC: {val_auc:.4f}")

    if val_auc > best_val_auc:
        print(f"  [+] AUC improved ({best_val_auc:.4f} -> {val_auc:.4f}). Saving model!")
        best_val_auc = val_auc
        torch.save(model.state_dict(), MODEL_SAVE_PATH)

print(f"\nTraining complete! Best Val ROC-AUC: {best_val_auc:.4f}")

## Inference on test soundscapes

In [ ]:
# Load best checkpoint
model.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=device))
model.to(device)
model.eval()
print("Best model loaded.")

TEST_AUDIO_DIR = os.path.join(BASE, "test_soundscapes")
test_files = glob.glob(os.path.join(TEST_AUDIO_DIR, "*.ogg"))

if len(test_files) == 0:
    # Fallback for local debugging — use a couple of train soundscapes
    TEST_AUDIO_DIR = os.path.join(BASE, "train_soundscapes")
    test_files = glob.glob(os.path.join(TEST_AUDIO_DIR, "*.ogg"))[:3]

print(f"Found {len(test_files)} test files.")

# Reuse dataset's mel/normalization pipeline for consistent preprocessing
dummy_ds = BirdCLEFDataset(
    pd.DataFrame([{'filename': '', 'audio_path': '', 'all_labels': [], 'is_soundscape': 0, 'start_sec': 0}]),
    Config, is_train=False
)

mel_transform  = dummy_ds.mel_transform.to(device)
amp_to_db      = dummy_ds.amp_to_db.to(device)

def chunk_to_mel(chunk):
    """Process a waveform chunk to a normalised 3-channel mel spectrogram."""
    mel = mel_transform(chunk)
    mel = amp_to_db(mel)
    # Robust normalisation (same as training)
    flat   = mel.flatten()
    median = flat.median()
    q75    = torch.quantile(flat, 0.75)
    q25    = torch.quantile(flat, 0.25)
    iqr    = q75 - q25 + 1e-6
    mel    = (mel - median) / iqr
    mel    = mel.clamp(-4, 4)
    mel    = mel.expand(3, -1, -1).unsqueeze(0)   # (1, 3, n_mels, T)
    return mel

chunk_length = Config.MAX_LENGTH   # 160 000 samples = 5 s
predictions  = []

with torch.no_grad():
    for file_path in tqdm(test_files, desc="Inference"):
        filename = os.path.basename(file_path)
        file_id  = filename.replace('.ogg', '')

        waveform, sr = torchaudio.load(file_path)
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)
        if sr != Config.SR:
            waveform = torchaudio.functional.resample(waveform, sr, Config.SR)
        waveform = waveform.to(device)

        total_samples = waveform.shape[1]
        num_chunks    = max(1, total_samples // chunk_length)

        for i in range(num_chunks):
            start = i * chunk_length
            end   = start + chunk_length
            chunk = waveform[:, start:end]

            # Pad if last chunk is shorter than 5 s
            if chunk.shape[1] < chunk_length:
                pad = chunk_length - chunk.shape[1]
                chunk = torch.nn.functional.pad(chunk, (0, pad))

            mel_inp = chunk_to_mel(chunk)
            logits  = model(mel_inp)
            probs   = torch.sigmoid(logits).cpu().numpy()[0]

            end_time = (i + 1) * 5
            row_id   = f"{file_id}_{end_time}"

            pred_dict = {'row_id': row_id}
            for class_name, prob in zip(CLASSES, probs):
                pred_dict[class_name] = float(prob)
            predictions.append(pred_dict)

submission_df = pd.DataFrame(predictions)
submission_path = os.path.join(WORK_DIR, 'submission.csv')
submission_df.to_csv(submission_path, index=False)
print(f"Submission saved to: {submission_path}")
print(f"Shape: {submission_df.shape}")
submission_df.head()